# **Feature Engineering**

This notebook is dedicated to conducting interactive experiments, statistical validation, and feature importance analysis for newly derived domain and interaction features. The goal is to uncover high-value predictive features that significantly boost model performance, before productionizing them into modular scripts (such as `src/features/builder.py`).

## 1. Import Required Libraries & Modules

In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

import seaborn as sns

from pathlib import Path
import sys 

notebook_dir = Path.cwd()
project_root = notebook_dir.parent 

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TRAIN_DATA_PATH, TEST_DATA_PATH
from src.preprocessing import Data_Preprocessor


## 2. Load Cleaned Datasets via Preprocessor

In [7]:
train_df_processed = Data_Preprocessor(expected_columns=21, is_train=True)
train_processed = train_df_processed.process_pipeline(str(TRAIN_DATA_PATH))

test_df_processed = Data_Preprocessor(expected_columns=20, is_train=False)
test_processed = test_df_processed.process_pipeline(str(TEST_DATA_PATH))

print(f"train shape {train_processed.shape}")
print(f"test shape {test_processed.shape}")

train shape (18478, 21)
test shape (6169, 20)


In [8]:
train_processed

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz,cost_category
0,tour_id1hffseyw,ITALY,45-64,With Children,0.0,2.0,Visiting Friends and Relatives,Beach Tourism,"Friends, relatives",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
1,tour_idnacd7zag,UNITED KINGDOM,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
2,tour_id62vz7e71,UNITED STATES OF AMERICA,65+,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,Yes,Yes,No,6,6,Yes,Higher Cost
3,tour_idrc76tzix,RWANDA,25-44,With Spouse and Children,3.0,1.0,Leisure and Holidays,Beach Tourism,"Radio, TV, Web",Independent,...,No,No,No,No,No,No,3,0,No,Lower Cost
4,tour_idn723m0n9,UNITED STATES OF AMERICA,45-64,Alone,0.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,Yes,Yes,7,0,Yes,Higher Cost
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18473,tour_idmp5ciw50,KENYA,45-64,Alone,0.0,1.0,Other,Hunting Tourism,Others,Independent,...,No,No,No,No,No,No,3,0,No,Lower Cost
18474,tour_ideq0yncfs,KENYA,45-64,Alone,1.0,0.0,Meetings and Conference,Wildlife Tourism,Others,Independent,...,No,No,No,No,No,No,2,0,No,Lower Cost
18475,tour_idv7pz3vs8,UNITED STATES OF AMERICA,25-44,With Spouse and Children,2.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Independent,...,No,No,No,No,No,No,9,0,Yes,Higher Cost
18476,tour_idy6ydo00w,UNITED STATES OF AMERICA,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Conference Tourism,"Radio, TV, Web",Package Tour,...,Yes,Yes,Yes,Yes,Yes,No,13,4,No,Higher Cost


In [9]:
train_processed.columns

Index(['Tour_ID', 'country', 'age_group', 'travel_with', 'total_female',
       'total_male', 'purpose', 'main_activity', 'info_source',
       'tour_arrangement', 'package_transport_int', 'package_accomodation',
       'package_food', 'package_transport_tz', 'package_sightseeing',
       'package_guided_tour', 'package_insurance', 'night_mainland',
       'night_zanzibar', 'first_trip_tz', 'cost_category'],
      dtype='object')

## **3. Feature Engineering**

### 3.1 Total nights.

In [10]:
train_processed['total nights'] = train_processed.night_mainland + train_processed.night_zanzibar

  ### 3.2 Total adults.

In [11]:
train_processed['total_adults'] = train_processed.total_female + train_processed.total_male

  ### 3.3 Night ratio.


In [12]:
# train_processed['night_ratio'] = train_processed['night_mainland'] / train_processed['night_zanzibar']

# to solve the division by zero problem. 
train_processed['night_ratio'] = np.where(
    train_processed['night_zanzibar'] == 0, 0,
    train_processed['night_mainland'] / train_processed['night_zanzibar']
)
train_processed['night_ratio'] = train_processed['night_ratio'].round(2)


  ### 3.4 Adults per night ratio.

In [13]:
train_processed['adults_per_night'] = train_processed['total_adults'] / train_processed['total nights']

train_processed['adults_per_night'] = train_processed['adults_per_night'].round(2)

  ### 3.5 Male to female ratio

In [14]:
train_processed['male_to_female_ratio'] = np.where(
    train_processed['total_female'] == 0, 0,
    train_processed['total_male'] / train_processed['total_female']
    )
train_processed['male_to_female_ratio'] = train_processed['male_to_female_ratio'].round(2)


  ### 3.6 Continent

In [15]:
country_dic = dict(enumerate(train_processed['country'].unique()))

In [16]:
country_dic

{0: 'ITALY',
 1: 'UNITED KINGDOM',
 2: 'UNITED STATES OF AMERICA',
 3: 'RWANDA',
 4: 'FRANCE',
 5: 'SWITZERLAND',
 6: 'NIGERIA',
 7: 'AUSTRALIA',
 8: 'SPAIN',
 9: 'DENMARK',
 10: 'BELGIUM',
 11: 'ZAMBIA',
 12: 'NETHERLANDS',
 13: 'CANADA',
 14: 'INDIA',
 15: 'NORWAY',
 16: 'ZIMBABWE',
 17: 'MOROCCO',
 18: 'CHINA',
 19: 'GERMANY',
 20: 'SOUTH AFRICA',
 21: 'JAPAN',
 22: 'UAE',
 23: 'RUSSIA',
 24: 'KENYA',
 25: 'GREECE',
 26: 'POLAND',
 27: 'AUSTRIA',
 28: 'MALAYSIA',
 29: 'CZECH REPUBLIC',
 30: 'KOREA',
 31: 'DRC',
 32: 'CYPRUS',
 33: 'KUWAIT',
 34: 'BRAZIL',
 35: 'CHILE',
 36: 'BULGARIA',
 37: 'ISRAEL',
 38: 'NEW ZEALAND',
 39: 'UGANDA',
 40: 'IRELAND',
 41: 'SWEDEN',
 42: 'QATAR',
 43: 'CONGO',
 44: 'BURUNDI',
 45: 'FINLAND',
 46: 'GHANA',
 47: 'OMAN',
 48: 'LEBANON',
 49: 'LUXEMBOURG',
 50: 'IVORY COAST',
 51: 'NAMIBIA',
 52: 'ROMANIA',
 53: 'THAILAND',
 54: 'SUDAN',
 55: 'SINGAPORE',
 56: 'PORTUGAL',
 57: 'MALAWI',
 58: 'MALTA',
 59: 'EGYPT',
 60: 'PAKISTAN',
 61: 'TANZANIA',
 62: '

In [17]:
continent_mapping = {
    'ITALY': 'EUROPE', 'UNITED KINGDOM': 'EUROPE', 'UNITED STATES OF AMERICA': 'NORTH AMERICA',
    'RWANDA': 'AFRICA', 'FRANCE': 'EUROPE', 'SWITZERLAND': 'EUROPE', 'NIGERIA': 'AFRICA',
    'AUSTRALIA': 'OCEANIA', 'SPAIN': 'EUROPE', 'DENMARK': 'EUROPE', 'BELGIUM': 'EUROPE',
    'ZAMBIA': 'AFRICA', 'NETHERLANDS': 'EUROPE', 'CANADA': 'NORTH AMERICA', 'INDIA': 'ASIA',
    'NORWAY': 'EUROPE', 'ZIMBABWE': 'AFRICA', 'MOROCCO': 'AFRICA', 'CHINA': 'ASIA',
    'GERMANY': 'EUROPE', 'SOUTH AFRICA': 'AFRICA', 'JAPAN': 'ASIA', 'UAE': 'ASIA',
    'RUSSIA': 'EUROPE/ASIA', 'KENYA': 'AFRICA', 'GREECE': 'EUROPE', 'POLAND': 'EUROPE',
    'AUSTRIA': 'EUROPE', 'MALAYSIA': 'ASIA', 'CZECH REPUBLIC': 'EUROPE', 'KOREA': 'ASIA',
    'DRC': 'AFRICA', 'CYPRUS': 'EUROPE', 'KUWAIT': 'ASIA', 'BRAZIL': 'SOUTH AMERICA',
    'CHILE': 'SOUTH AMERICA', 'BULGARIA': 'EUROPE', 'ISRAEL': 'ASIA', 'NEW ZEALAND': 'OCEANIA',
    'UGANDA': 'AFRICA', 'IRELAND': 'EUROPE', 'SWEDEN': 'EUROPE', 'QATAR': 'ASIA',
    'CONGO': 'AFRICA', 'BURUNDI': 'AFRICA', 'FINLAND': 'EUROPE', 'GHANA': 'AFRICA',
    'OMAN': 'ASIA', 'LEBANON': 'ASIA', 'LUXEMBOURG': 'EUROPE', 'IVORY COAST': 'AFRICA',
    'NAMIBIA': 'AFRICA', 'ROMANIA': 'EUROPE', 'THAILAND': 'ASIA', 'SUDAN': 'AFRICA',
    'SINGAPORE': 'ASIA', 'PORTUGAL': 'EUROPE', 'MALAWI': 'AFRICA', 'MALTA': 'EUROPE',
    'EGYPT': 'AFRICA', 'PAKISTAN': 'ASIA', 'TANZANIA': 'AFRICA', 'ARGENTINA': 'SOUTH AMERICA',
    'SRI LANKA': 'ASIA', 'BOTSWANA': 'AFRICA', 'VIETNAM': 'ASIA', 'BERMUDA': 'NORTH AMERICA',
    'TAIWAN': 'ASIA', 'SENEGAL': 'AFRICA', 'AFGHANISTAN': 'ASIA', 'COLOMBIA': 'SOUTH AMERICA',
    'JAMAICA': 'NORTH AMERICA', 'BAHRAIN': 'ASIA', 'COSTA RICA': 'NORTH AMERICA', 'COMOROS': 'AFRICA',
    'URUGUAY': 'SOUTH AMERICA', 'SAUDI ARABIA': 'ASIA', 'ETHIOPIA': 'AFRICA', 'PHILIPPINES': 'ASIA',
    'ARMENIA': 'ASIA', 'MEXICO': 'NORTH AMERICA', 'MADAGASCAR': 'AFRICA', 'SERBIA': 'EUROPE',
    'SLOVAKIA': 'EUROPE', 'IRAN': 'ASIA', 'SWAZILAND': 'AFRICA', 'PERU': 'SOUTH AMERICA',
    'LITHUANIA': 'EUROPE', 'ALGERIA': 'AFRICA', 'SEYCHELLES': 'AFRICA', 'MOZAMBIQUE': 'AFRICA',
    'UKRAINE': 'EUROPE', 'NEPAL': 'ASIA', 'CROATIA': 'EUROPE', 'SOMALIA': 'AFRICA',
    'LIBERIA': 'AFRICA', 'MAURITIUS': 'AFRICA', 'GAMBIA': 'AFRICA', 'ECUADOR': 'SOUTH AMERICA',
    'LATVIA': 'EUROPE', 'MACEDONIA': 'EUROPE', 'LESOTHO': 'AFRICA', 'HUNGARY': 'EUROPE',
    'ICELAND': 'EUROPE', 'VENEZUELA': 'SOUTH AMERICA', 'ANGOLA': 'AFRICA', 'SLOVENIA': 'EUROPE',
    'CAPE VERDE': 'AFRICA', 'CAMEROON': 'AFRICA', 'MONECASQUE': 'EUROPE', 'TRINIDAD AND TOBAGO': 'SOUTH AMERICA',
    'PAPUA NEW GUINEA': 'OCEANIA', 'TURKEY': 'EUROPE/ASIA', 'INDONESIA': 'ASIA', 'ESTONIA': 'EUROPE',
    'BOSNIA': 'EUROPE', 'YEMEN': 'ASIA', 'NIGER': 'AFRICA', 'ERITREA': 'AFRICA',
    'GEORGIA': 'ASIA', 'BANGLADESH': 'ASIA', 'BARBADOS': 'NORTH AMERICA', 'TUNISIA': 'AFRICA',
    'DJIBOUTI': 'AFRICA', 'DOMINICA': 'NORTH AMERICA', 'CAMBODIA': 'ASIA', 'MONTENEGRO': 'EUROPE',
    'JORDAN': 'ASIA'
}

In [18]:
train_processed['continent'] = train_processed['country'].map(continent_mapping)

In [19]:
train_processed

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,night_mainland,night_zanzibar,first_trip_tz,cost_category,total nights,total_adults,night_ratio,adults_per_night,male_to_female_ratio,continent
0,tour_id1hffseyw,ITALY,45-64,With Children,0.0,2.0,Visiting Friends and Relatives,Beach Tourism,"Friends, relatives",Package Tour,...,0,7,Yes,High Cost,7,2.0,0.00,0.29,0.00,EUROPE
1,tour_idnacd7zag,UNITED KINGDOM,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,0,7,Yes,High Cost,7,2.0,0.00,0.29,1.00,EUROPE
2,tour_id62vz7e71,UNITED STATES OF AMERICA,65+,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,6,6,Yes,Higher Cost,12,2.0,1.00,0.17,1.00,NORTH AMERICA
3,tour_idrc76tzix,RWANDA,25-44,With Spouse and Children,3.0,1.0,Leisure and Holidays,Beach Tourism,"Radio, TV, Web",Independent,...,3,0,No,Lower Cost,3,4.0,0.00,1.33,0.33,AFRICA
4,tour_idn723m0n9,UNITED STATES OF AMERICA,45-64,Alone,0.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,7,0,Yes,Higher Cost,7,1.0,0.00,0.14,0.00,NORTH AMERICA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18473,tour_idmp5ciw50,KENYA,45-64,Alone,0.0,1.0,Other,Hunting Tourism,Others,Independent,...,3,0,No,Lower Cost,3,1.0,0.00,0.33,0.00,AFRICA
18474,tour_ideq0yncfs,KENYA,45-64,Alone,1.0,0.0,Meetings and Conference,Wildlife Tourism,Others,Independent,...,2,0,No,Lower Cost,2,1.0,0.00,0.50,0.00,AFRICA
18475,tour_idv7pz3vs8,UNITED STATES OF AMERICA,25-44,With Spouse and Children,2.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Independent,...,9,0,Yes,Higher Cost,9,3.0,0.00,0.33,0.50,NORTH AMERICA
18476,tour_idy6ydo00w,UNITED STATES OF AMERICA,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Conference Tourism,"Radio, TV, Web",Package Tour,...,13,4,No,Higher Cost,17,2.0,3.25,0.12,1.00,NORTH AMERICA


In [20]:
train_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18478 entries, 0 to 18477
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                18478 non-null  object 
 1   country                18478 non-null  object 
 2   age_group              18478 non-null  object 
 3   travel_with            18478 non-null  object 
 4   total_female           18478 non-null  float64
 5   total_male             18478 non-null  float64
 6   purpose                18478 non-null  object 
 7   main_activity          18478 non-null  object 
 8   info_source            18478 non-null  object 
 9   tour_arrangement       18478 non-null  object 
 10  package_transport_int  18478 non-null  object 
 11  package_accomodation   18478 non-null  object 
 12  package_food           18478 non-null  object 
 13  package_transport_tz   18478 non-null  object 
 14  package_sightseeing    18478 non-null  object 
 15  pa

## Feature Engineering Summary 
